In [ ]:
# !pip install pandas sentence-transformers faiss-cpu rank-bm25 transformers accelerate
# !pip install bertopic umap-learn hdbscan
# !pip install openpyxl


In [ ]:
# Standard library imports
import ast
import itertools
import os
import pickle
import random
import re
import textwrap
from difflib import SequenceMatcher
from pathlib import Path

# Third-party library imports
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hdbscan import HDBSCAN

# Scikit-learn imports
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import (
    CountVectorizer, 
    TfidfVectorizer, 
    ENGLISH_STOP_WORDS
)

# NLP and Topic Modeling imports
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Jupyter/IPython display and widgets
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [ ]:


# === GLOBAL CONFIG ===
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA_DIR = Path(".")
INDEX_DIR = DATA_DIR / "index"
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # embedding model for retrieval/topic models
LOCAL_LLM_NAME = "google/flan-t5-large"  # local seq2seq model for generation
DEFAULT_TOP_K = 6

# === FILE PATHS ===
INPUT_CSVS = sorted(DATA_DIR.glob('FGD*_transcript_turns.csv'))
OUTPUT_CSV_CLEAN = "FGD_all_transcript_turns_cleaned.csv"

# Preview settings for console output
CLEAN_SAMPLE_ROWS = 10

assert INPUT_CSVS, "No input CSVs found matching pattern FGD*_transcript_turns.csv"

# === LOAD DATA FROM CSVs ===
frames = []
for csv_path in INPUT_CSVS:
    print(f"Loading {csv_path.name} ...")
    df_part = pd.read_csv(csv_path)
    df_part["source_file"] = csv_path.name
    frames.append(df_part)

df = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(INPUT_CSVS)} files.")
print("Columns found:", df.columns.tolist())
print("Number of rows before cleaning:", len(df))

# Make sure we actually have an 'utterance' column
assert "utterance" in df.columns, "Expected an 'utterance' column in the input CSVs."


# ---------- 1. Helper: remove filler words ----------

FILLER_PATTERNS = [
    r"\buh\b",
    r"\bum\b",
    r"\berm\b",
    r"\ber\b",
    r"\burr\b",
    r"\bah\b",
    r"\buhm\b",
    r"\bmmm\b",
    r"\byou know\b",
    r"\bkind of\b",
    r"\bsort of\b",
    r"\bi mean\b",
    r"\bi guess\b",
    r"\bright\?\b",
    r"\bok,\b",
    r"\bok.\b",
    r"\boh,\b",
    r"\bso,\b",
    r"\bok\b",
    r"\bokay\b",
    r"\bwell,\b",
]

FILLER_REGEXES = [re.compile(pat, flags=re.IGNORECASE) for pat in FILLER_PATTERNS]


def remove_filler_words(text: str) -> str:
    if not isinstance(text, str):
        return text
    new_text = text
    for regex in FILLER_REGEXES:
        new_text = regex.sub(" ", new_text)
    return new_text


# ---------- 2. Helper: collapse repeated words ----------

def collapse_repeated_words(text: str) -> str:
    """
    Turns:
        "OK OK OK, I think..." -> "OK, I think..."
        "ya ya" -> "ya"
    Only for directly repeated words.
    """
    if not isinstance(text, str):
        return text

    pattern = re.compile(r"\b(\w+)(\s+\1\b)+", flags=re.IGNORECASE)
    return pattern.sub(r"\1", text)


# ---------- 3. Master cleaning function for utterances ----------

def clean_utterance(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = remove_filler_words(text)
    text = collapse_repeated_words(text)


    # Punctuation tidy: drop leading stray punct (even after quotes), trim isolated commas/periods, collapse repeats, drop trailing lone period
    text = re.sub(r"^[^A-Za-z0-9]+(?=[A-Za-z0-9])", "", text)
    text = re.sub(r"^[\"\']*[.,!?]+\s*", "", text)
    text = re.sub(r"\s+[,\.]{1,}\s+", " ", text)
    text = re.sub(r"([,\.\.!?…]){2,}", r"\1", text)
    text = re.sub(r"\s*\.$", "", text)  # drop trailing lone period like "word ."

    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------- 4. Apply cleaning pipeline (no row removals) ----------

# Keep original utterance for audit if not already present
if "utterance_original" not in df.columns:
    df["utterance_original"] = df["utterance"]

# Clean the utterance text in place
df["utterance"] = df["utterance"].apply(clean_utterance)

# Turn IDs stay sequential by session
if "session_id" in df.columns:
    if "turn_id" in df.columns:
        df = df.sort_values(["session_id", "turn_id"]).reset_index(drop=True)
    else:
        df = df.sort_values(["session_id"]).reset_index(drop=True)
    df["turn_id"] = df.groupby("session_id").cumcount() + 1
else:
    df = df.reset_index(drop=True)
    df["turn_id"] = df.index + 1

print("Number of rows after cleaning (no removals):", len(df))
print(f"\nSample of cleaned output (first {CLEAN_SAMPLE_ROWS} rows):")
print(df.head(CLEAN_SAMPLE_ROWS))

# ---------- 5. Save cleaned data ----------

df.to_csv(OUTPUT_CSV_CLEAN, index=False, encoding="utf-8-sig")
print(f"\nCleaned data saved to: {OUTPUT_CSV_CLEAN}")


In [ ]:


INPUT_CSV_CLEAN = OUTPUT_CSV_CLEAN

df = pd.read_csv(DATA_DIR / INPUT_CSV_CLEAN)
print(df.columns)
print(len(df))

# Basic sanity: assume we have columns: session_id, turn_id, speaker_name, speaker_role, utterance
df = df.sort_values(["session_id", "turn_id"]).reset_index(drop=True)


# ---- Chunking helper ----
def make_chunks(df, max_chars=1000, overlap_chars=150):
    """
    Greedy chunking by turn, keeping turns for context.
    max_chars is per chunk; overlap_chars is how much text to repeat between chunks.
    """
    chunks = []
    current = ""
    current_meta = []
    current_session = None

    for _, row in df.iterrows():
        row_session = row["session_id"]
        # If session changes, flush current chunk without overlap
        if current_session is None:
            current_session = row_session
        elif row_session != current_session:
            if current:
                chunks.append({
                    "session_id": current_meta[0]["session_id"],
                    "text": current.strip(),
                    "turn_start": current_meta[0]["turn_id"],
                    "turn_end": current_meta[-1]["turn_id"],
                })
            current = ""
            current_meta = []
            current_session = row_session

        text = str(row["utterance"])
        turn_info = (
            f'{row["speaker_name"]} ({row["speaker_role"]}) '
            f'[turn {row["turn_id"]}]: {text}\n'
        )

        if len(current) + len(turn_info) > max_chars and current:
            # save current chunk
            chunks.append({
                "session_id": current_meta[0]["session_id"],
                "text": current.strip(),
                "turn_start": current_meta[0]["turn_id"],
                "turn_end": current_meta[-1]["turn_id"],
            })
            # start new chunk with overlap
            overlap = current[-overlap_chars:]
            current = overlap + "\n" + turn_info
            current_meta = [row]
        else:
            current += turn_info
            current_meta.append(row)

    if current:
        chunks.append({
            "session_id": current_meta[0]["session_id"],
            "text": current.strip(),
            "turn_start": current_meta[0]["turn_id"],
            "turn_end": current_meta[-1]["turn_id"],
        })

    return pd.DataFrame(chunks)


chunks_df = make_chunks(df, max_chars=1000, overlap_chars=200)
# Stable chunk id for alignment with FAISS
chunks_df["chunk_id"] = chunks_df.apply(
    lambda r: f"s{r['session_id']}_t{r['turn_start']}-{r['turn_end']}", axis=1
)
print(chunks_df.head())
print("Num chunks:", len(chunks_df))


In [ ]:

# Export chunks to Excel
# Requires chunking cell to have run (chunks_df in memory)
if 'chunks_df' not in globals():
    raise RuntimeError("Run the chunking cell first to create chunks_df.")

export_path = "chunks_export.xlsx"
chunks_df.to_excel(export_path, index=False)
print(f"Saved chunks to: {export_path}")


In [ ]:
# Shared stopwords

custom_stop = {
    "doesn", "Said", "said", "Them", "them", "Rest", "rest", "Key", "key", "Far", "far", "End", "end", "Use", "use", "Make", 
    "make", "Sure", "sure", "different", "similar"

}

stop_words = list(ENGLISH_STOP_WORDS.union(custom_stop))


In [ ]:
# Shared prompt builder

def build_prompt(context, question):
    return (
        "You are a research analyst reviewing a focus group transcript.\n"
        "Use only the context to answer the question.\n"
        "Answer in 2–3 sentences and cite specific details from the context.\n"
        "If the answer is not in the context, say 'Not enough context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


In [ ]:
# Shared helpers


def ensure_retrieval():
    """Ensure search_chunks is defined; auto-load from saved index if possible."""
    global search_chunks, chunks_df
    if 'search_chunks' in globals():
        return
    emb_name_path = INDEX_DIR / 'fgd_all_emb_model_name.txt'
    faiss_path = INDEX_DIR / 'fgd_all.index'
    chunks_path = INDEX_DIR / 'fgd_all_chunks.pkl'
    if not (emb_name_path.exists() and faiss_path.exists() and chunks_path.exists()):
        raise RuntimeError("Run the retrieval cell first to define search_chunks.")
    import faiss
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import pandas as pd
    chunks_df = pd.read_pickle(chunks_path)
    emb_model_name = emb_name_path.read_text().strip()
    retrieval_model = SentenceTransformer(emb_model_name)
    faiss_index = faiss.read_index(str(faiss_path))

    def search_chunks(query, k=DEFAULT_TOP_K):
        q_emb = retrieval_model.encode([query], normalize_embeddings=True)
        D, I = faiss_index.search(q_emb, k)
        results = []
        for score, idx in zip(D[0], I[0]):
            row = chunks_df.iloc[idx]
            cid = row.get('chunk_id') or f"s{row['session_id']}_t{row['turn_start']}-{row['turn_end']}"
            results.append({
                'chunk_id': cid,
                'score': float(score),
                'session_id': str(row.get('session_id', '')),
                'turn_start': int(row.get('turn_start', 0)),
                'turn_end': int(row.get('turn_end', 0)),
                'text': row.get('text', ''),
            })
        return results


def ensure_llm():
    """Load the local LLM if it is not already initialized."""
    global _gen
    if '_gen' in globals():
        return
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
    import torch
    _device_map = {"": "mps"} if torch.backends.mps.is_available() else {"": "cpu"}
    _dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
    _tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)
    _model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_LLM_NAME, device_map=_device_map, torch_dtype=_dtype)
    _gen = pipeline("text2text-generation", model=_model, tokenizer=_tokenizer, max_new_tokens=320)


In [ ]:
# retrieval system


EMB_DIM = 384  # dimension for this model

model = SentenceTransformer(EMB_MODEL_NAME)

# Encode all chunks
embeddings = model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embeddings shape:", embeddings.shape)

# Build FAISS index
index = faiss.IndexFlatIP(EMB_DIM)  # inner product (works with normalized vectors)
index.add(embeddings)
print("Index size:", index.ntotal)

# Save everything for later reuse
INDEX_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(INDEX_DIR / "fgd_all.index"))
chunks_df.to_pickle(INDEX_DIR / "fgd_all_chunks.pkl")
with open(INDEX_DIR / "fgd_all_emb_model_name.txt", "w") as f:
    f.write(EMB_MODEL_NAME)


In [ ]:
# N-gram keyphrase extraction (bigrams/trigrams)
# Extract top-scoring n-grams from chunk texts as theme hints
if 'stop_words' not in globals():
    raise RuntimeError("Run the shared stopwords cell first.")
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import re

# Clean texts
ngram_docs = [t.strip() for t in chunks_df["text"].fillna("").tolist() if str(t).strip()]
if not ngram_docs:
    print("No text to extract n-grams from.")
else:
    def try_vectorizer(cfg):
        vec = TfidfVectorizer(**cfg)
        return vec, vec.fit_transform(ngram_docs)

    configs = [
        {
            "ngram_range": (6, 6),
            "max_df": 0.85,
            "min_df": 2,
            "stop_words": list(stop_words),
            "token_pattern": r"(?u)\b[A-Za-z][\w]+\b",
            "preprocessor": lambda x: re.sub(r"\d+", " ", x),
        },
        {
            "ngram_range": (2, 3),
            "max_df": 1.0,
            "min_df": 1,
            "stop_words": None,
            "token_pattern": r"(?u)\b[A-Za-z][\w]+\b",
            "preprocessor": lambda x: re.sub(r"\d+", " ", x),
        },
        {
            "ngram_range": (1, 3),
            "max_df": 1.0,
            "min_df": 1,
            "stop_words": None,
            "token_pattern": r"(?u)\b[A-Za-z][\w]+\b",
            "preprocessor": lambda x: re.sub(r"\d+", " ", x),
        },
    ]

    tfidf = None
    vectorizer = None
    for cfg in configs:
        try:
            vectorizer, tfidf = try_vectorizer(cfg)
            print(f"TF-IDF succeeded with config: {cfg}")
            break
        except ValueError as e:
            print(f"TF-IDF failed with config {cfg}: {e}")
            tfidf = None
            vectorizer = None

    if tfidf is None or tfidf.shape[1] == 0:
        print("No n-gram vocabulary found; showing a sample doc for debugging:")
        print(ngram_docs[0][:400])
    else:
        terms = np.array(vectorizer.get_feature_names_out())
        scores = tfidf.sum(axis=0).A1
        top_n = 20
        top_idx = np.argsort(scores)[::-1][:top_n]
        ngram_df = pd.DataFrame({"ngram": terms[top_idx], "score": scores[top_idx]})
        print("Top n-grams (bigrams/trigrams):")
        print(ngram_df)


In [ ]:

# Export n-gram keyphrases to Excel
if 'ngram_df' not in globals() or ngram_df.empty:
    raise RuntimeError("Run the n-gram extraction cell first to create ngram_df.")
export_path = "ngram_keyphrases.xlsx"
ngram_df.to_excel(export_path, index=False)
print(f"Saved n-gram keyphrases to: {export_path}")


In [ ]:
# Topic modeling (LDA) on chunks_df
# Ensure chunks_df is loaded
if 'chunks_df' not in globals():
    try:
        chunks_df = pd.read_pickle(INDEX_DIR / 'fgd_all_chunks.pkl')
    except FileNotFoundError:
        print('chunks_df not found; run earlier cells first.')
        chunks_df = None

if chunks_df is not None:
    if 'stop_words' not in globals():
        raise RuntimeError("Run the shared stopwords cell first.")
    texts = chunks_df['text'].fillna('').tolist()
    if not texts:
        print('No text available for topic modeling.')
    else:
        vectorizer = CountVectorizer(
            max_df=0.95,
            min_df=2,
            stop_words=stop_words,
        )
        dtm = vectorizer.fit_transform(texts)

        n_topics = 8  # adjust as needed
        lda = LatentDirichletAllocation(
            n_components=n_topics,
            learning_method='batch',
            random_state=42,
        )
        lda_doc_topics = lda.fit_transform(dtm)

        terms = vectorizer.get_feature_names_out()
        topn = 10
        for idx, comp in enumerate(lda.components_):
            top_terms = [terms[i] for i in comp.argsort()[:-topn-1:-1]]
            print(f'Topic {idx}: ' + ', '.join(top_terms))

        # Assign dominant topic per chunk (for quick inspection)
        chunks_df['topic'] = lda_doc_topics.argmax(axis=1)
        print(chunks_df[['session_id', 'turn_start', 'turn_end', 'topic']].head())

# To do: combine with trigrams as currently only one word exist but can make it n words


In [ ]:

# Export LDA topics and chunk assignments to Excel
import pandas as pd
if 'lda' not in globals() or 'vectorizer' not in globals():
    raise RuntimeError("Run the LDA topic modeling cell first to create lda and vectorizer.")
if 'chunks_df' not in globals():
    chunks_df = pd.read_pickle('index/fgd_all_chunks.pkl')

# topics table
terms = vectorizer.get_feature_names_out()
topn = 10
topic_rows = []
for idx, comp in enumerate(lda.components_):
    top_terms = [terms[i] for i in comp.argsort()[:-topn-1:-1]]
    topic_rows.append({"topic": idx, "top_terms": ", ".join(top_terms)})
topic_df = pd.DataFrame(topic_rows)

# chunk assignments
if 'topic' not in chunks_df.columns:
    chunks_df['topic'] = lda.transform(vectorizer.transform(chunks_df['text'].fillna(''))).argmax(axis=1)
assign_df = chunks_df[['session_id','turn_start','turn_end','chunk_id','topic']]

export_path = "lda_topics.xlsx"
with pd.ExcelWriter(export_path) as writer:
    topic_df.to_excel(writer, sheet_name="topics", index=False)
    assign_df.to_excel(writer, sheet_name="chunk_topics", index=False)
print(f"Saved LDA topics and assignments to: {export_path}")


In [ ]:
# Transformer-based topic modeling (BERTopic)
# Requires: bertopic, sentence-transformers, umap-learn, hdbscan
# If needed, install with: !pip install bertopic umap-learn hdbscan

# Ensure chunks_df is available
if 'chunks_df' not in globals():
    import pandas as pd
    chunks_df = pd.read_pickle(INDEX_DIR / 'fgd_all_chunks.pkl')

texts = chunks_df['text'].fillna('').tolist()
if not texts:
    raise RuntimeError('No text available for BERTopic.')

# Use the same embedding model as retrieval (can change to a stronger one if desired)
embedding_model = SentenceTransformer(EMB_MODEL_NAME)

# Custom stop words (reuse your list)
if 'stop_words' not in globals():
    raise RuntimeError("Run the shared stopwords cell first.")
vectorizer_model = CountVectorizer(stop_words=stop_words)

# Increase topic granularity
hdbscan_model = HDBSCAN(min_cluster_size=5)

# Fit BERTopic
bertopic = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    nr_topics=10,
    verbose=True,
)
topics, probs = bertopic.fit_transform(texts)

# Show top words per topic
topic_info = bertopic.get_topic_info().head(10)
print(topic_info)

for t in topic_info['Topic'].tolist():
    if t == -1:
        continue
    print(f"Topic {t}:", bertopic.get_topic(t))


In [ ]:

# Export BERTopic topic info to Excel
# Requires bertopic cell to have run
if 'bertopic' not in globals():
    raise RuntimeError("Run the BERTopic cell first to create the model.")

export_path = "bertopic_topics.xlsx"
topic_info = bertopic.get_topic_info()
topic_info.to_excel(export_path, index=False)
print(f"Saved BERTopic topic info to: {export_path}")


In [ ]:
# Test summary of methods - N-gram
ensure_retrieval()
if '_gen' not in globals():
    ensure_llm()
if 'ngram_df' not in globals() or getattr(ngram_df, 'empty', True):
    raise RuntimeError("Run the n-gram extraction cell first.")

TOP_NGRAMS = 5
for ng in ngram_df['ngram'].head(TOP_NGRAMS).tolist():
    query = f"Summarize the discussion about '{ng}' in 2-3 sentences."
    hits = search_chunks(query, k=6)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, query)
    summary = _gen(prompt)[0]['generated_text']
    print("=" * 80)
    print("N-gram:", ng)
    print("Summary:", summary)
    print("Sources:", ", ".join(h["chunk_id"] for h in hits))
    general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about {ng}."
    general_summary = _gen(general_prompt)[0]['generated_text']
    print("General summary (LLM, not limited to transcript):", general_summary)

In [ ]:
# Test summary of methods - LDA
ensure_retrieval()
if '_gen' not in globals():
    ensure_llm()
if 'lda' not in globals() or 'vectorizer' not in globals():
    raise RuntimeError("Run the LDA topic modeling cell first.")

terms = vectorizer.get_feature_names_out()
TOP_TERMS = 6
for t in range(lda.n_components):
    comp = lda.components_[t]
    top_terms = [terms[i] for i in comp.argsort()[:-TOP_TERMS-1:-1]]
    term_str = ", ".join(top_terms)
    query = f"Summarize the discussion around these keywords: {term_str}. Keep it to 2-3 sentences."
    hits = search_chunks(query, k=6)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, query)
    summary = _gen(prompt)[0]['generated_text']
    print("=" * 80)
    print(f"LDA Topic {t} terms:", term_str)
    print("Summary:", summary)
    print("Sources:", ", ".join(h["chunk_id"] for h in hits))
    general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about these keywords: {term_str}."
    general_summary = _gen(general_prompt)[0]['generated_text']
    print("General summary (LLM, not limited to transcript):", general_summary)

In [ ]:
# Test summary of methods - BERTopic
ensure_retrieval()
if '_gen' not in globals():
    ensure_llm()
if 'bertopic' not in globals():
    raise RuntimeError("Run the BERTopic cell first.")

MAX_TOPICS = 5
TOP_TERMS = 6
topic_info = bertopic.get_topic_info()
valid_topics = [t for t in topic_info['Topic'].tolist() if t != -1][:MAX_TOPICS]
for t in valid_topics:
    terms = [w for w, _ in bertopic.get_topic(t)][:TOP_TERMS]
    term_str = ", ".join(terms)
    query = f"Summarize the discussion around these keywords: {term_str}. Keep it to 2-3 sentences."
    hits = search_chunks(query, k=6)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, query)
    summary = _gen(prompt)[0]['generated_text']
    print("=" * 80)
    print(f"BERTopic {t} terms:", term_str)
    print("Summary:", summary)
    print("Sources:", ", ".join(h["chunk_id"] for h in hits))
    general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about these keywords: {term_str}."
    general_summary = _gen(general_prompt)[0]['generated_text']
    print("General summary (LLM, not limited to transcript):", general_summary)

In [ ]:

# NEW CELL TO TEST
# Topic/phrase distinctiveness and coverage metrics for LDA, BERTopic, and n-grams

TOPN_TERMS = 10  # top terms/phrases per topic to evaluate

# Helpers

def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def topic_diversity(topic_term_lists):
    all_terms = list(itertools.chain.from_iterable(topic_term_lists))
    if not all_terms:
        return 0.0
    return len(set(all_terms)) / len(all_terms)

def avg_pairwise_jaccard(topic_term_lists):
    pairs = list(itertools.combinations(topic_term_lists, 2))
    if not pairs:
        return 0.0
    scores = [jaccard(a, b) for a, b in pairs]
    return float(np.mean(scores)) if scores else 0.0

def doc_contains_any(text, phrases):
    t = str(text).lower()
    return any(p.lower() in t for p in phrases)

# 1) LDA metrics
if 'lda' in globals() and 'vectorizer' in globals():
    terms = vectorizer.get_feature_names_out()
    lda_topics = []
    for comp in lda.components_:
        top_idx = comp.argsort()[:-TOPN_TERMS-1:-1]
        lda_topics.append([terms[i] for i in top_idx])
    lda_diversity = topic_diversity(lda_topics)
    lda_overlap = avg_pairwise_jaccard(lda_topics)
    lda_coverage = None
    if 'chunks_df' in globals():
        texts = chunks_df['text'].fillna('').tolist()
        lda_phrases = set(itertools.chain.from_iterable(lda_topics))
        hits = sum(doc_contains_any(t, lda_phrases) for t in texts)
        lda_coverage = hits / len(texts) if texts else 0.0
    print("LDA -> diversity={:.3f}, avg_jaccard={:.3f}, coverage={}".format(
        lda_diversity, lda_overlap, f"{lda_coverage:.3f}" if lda_coverage is not None else 'n/a'))
else:
    print("LDA metrics skipped (lda/vectorizer missing)")

# 2) BERTopic metrics
if 'bertopic' in globals():
    topic_info = bertopic.get_topic_info()
    topic_ids = [t for t in topic_info['Topic'].tolist() if t != -1]
    bert_topics = []
    for t in topic_ids:
        words = [w for w, _ in bertopic.get_topic(t)][:TOPN_TERMS]
        bert_topics.append(words)
    bert_diversity = topic_diversity(bert_topics)
    bert_overlap = avg_pairwise_jaccard(bert_topics)
    bert_coverage = None
    if 'chunks_df' in globals():
        texts = chunks_df['text'].fillna('').tolist()
        bert_phrases = set(itertools.chain.from_iterable(bert_topics))
        hits = sum(doc_contains_any(t, bert_phrases) for t in texts)
        bert_coverage = hits / len(texts) if texts else 0.0
    print("BERTopic -> diversity={:.3f}, avg_jaccard={:.3f}, coverage={}".format(
        bert_diversity, bert_overlap, f"{bert_coverage:.3f}" if bert_coverage is not None else 'n/a'))
else:
    print("BERTopic metrics skipped (bertopic missing)")

# 3) N-gram metrics (uses top phrases only)
if 'ngram_df' in globals() and not getattr(ngram_df, 'empty', True):
    top_ngrams = ngram_df['ngram'].head(TOPN_TERMS).tolist()
    # Treat each n-gram as its own "topic" of one phrase for overlap/diversity
    ngram_topics = [[ng] for ng in top_ngrams]
    ngram_diversity = topic_diversity(ngram_topics)
    ngram_overlap = avg_pairwise_jaccard(ngram_topics)
    ngram_coverage = None
    if 'chunks_df' in globals():
        texts = chunks_df['text'].fillna('').tolist()
        hits = sum(doc_contains_any(t, top_ngrams) for t in texts)
        ngram_coverage = hits / len(texts) if texts else 0.0
    print("N-gram -> diversity={:.3f}, avg_jaccard={:.3f}, coverage={}".format(
        ngram_diversity, ngram_overlap, f"{ngram_coverage:.3f}" if ngram_coverage is not None else 'n/a'))
else:
    print("N-gram metrics skipped (ngram_df missing/empty)")


In [ ]:

# Auto Search
# Lightweight hyperparameter sweep for topic distinctiveness/coverage metrics

# Controls (tune as desired)
LDA_TRIALS = 5
NGRAM_TRIALS = 5
BERTOPIC_TRIALS = 2  # set >0 to try BERTopic reductions (can be slower)
TOPN_TERMS = 10

# Text corpus
docs = chunks_df['text'].fillna('').tolist() if 'chunks_df' in globals() else []
if not docs:
    raise RuntimeError("chunks_df with text is required for auto search.")

# Metrics helpers
def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def topic_diversity(topic_term_lists):
    all_terms = list(itertools.chain.from_iterable(topic_term_lists))
    return len(set(all_terms)) / len(all_terms) if all_terms else 0.0

def avg_pairwise_jaccard(topic_term_lists):
    pairs = list(itertools.combinations(topic_term_lists, 2))
    if not pairs:
        return 0.0
    scores = [jaccard(a, b) for a, b in pairs]
    return float(np.mean(scores)) if scores else 0.0

def doc_contains_any(text, phrases):
    t = str(text).lower()
    return any(p.lower() in t for p in phrases)

def eval_topics(topic_term_lists, texts):
    diversity = topic_diversity(topic_term_lists)
    overlap = avg_pairwise_jaccard(topic_term_lists)
    phrases = set(itertools.chain.from_iterable(topic_term_lists))
    coverage = sum(doc_contains_any(t, phrases) for t in texts)
    coverage = coverage / len(texts) if texts else 0.0
    # Simple objective: maximize diversity + coverage - overlap
    objective = diversity + coverage - overlap
    return {
        'diversity': diversity,
        'overlap': overlap,
        'coverage': coverage,
        'objective': objective,
    }

results = []

# --- LDA sweep ---
lda_configs = []
for _ in range(LDA_TRIALS):
    lda_configs.append({
        'n_components': random.choice([6, 8, 10, 12, 14]),
        'max_df': random.choice([0.7, 0.8, 0.9, 0.95]),
        'min_df': random.choice([1, 2, 3]),
        'ngram_range': random.choice([(1, 1), (1, 2)]),
        'eta': random.choice(['auto', 'auto']),
    })

for cfg in lda_configs:
    try:
        vec = CountVectorizer(max_df=cfg['max_df'], min_df=cfg['min_df'], ngram_range=cfg['ngram_range'])
        dtm = vec.fit_transform(docs)
        if dtm.shape[1] == 0:
            continue
        lda = LatentDirichletAllocation(
            n_components=cfg['n_components'],
            learning_method='batch',
            max_iter=10,
            random_state=42,
            doc_topic_prior=None,
            topic_word_prior=None if cfg['eta'] == 'auto' else cfg['eta'],
        )
        lda.fit(dtm)
        terms = vec.get_feature_names_out()
        topic_terms = []
        for comp in lda.components_:
            idx = comp.argsort()[:-TOPN_TERMS-1:-1]
            topic_terms.append([terms[i] for i in idx])
        metrics = eval_topics(topic_terms, docs)
        results.append(('LDA', metrics['objective'], metrics, cfg))
    except Exception as e:
        print("LDA config failed", cfg, e)

# --- N-gram sweep ---
ngram_configs = []
for _ in range(NGRAM_TRIALS):
    ngram_configs.append({
        'ngram_range': random.choice([(1, 2), (1, 3)]),
        'max_df': random.choice([0.8, 0.9, 1.0]),
        'min_df': random.choice([1, 2]),
        'top_k': random.choice([10, 15, 20, 30]),
    })

for cfg in ngram_configs:
    try:
        vec = TfidfVectorizer(
            ngram_range=cfg['ngram_range'],
            max_df=cfg['max_df'],
            min_df=cfg['min_df'],
            token_pattern=r"(?u)\b[A-Za-z][\w]+\b",
            lowercase=True,
        )
        tfidf = vec.fit_transform(docs)
        if tfidf.shape[1] == 0:
            continue
        scores = tfidf.sum(axis=0).A1
        terms = vec.get_feature_names_out()
        top_idx = scores.argsort()[::-1][:cfg['top_k']]
        top_terms = [terms[i] for i in top_idx]
        topic_terms = [[t] for t in top_terms]
        metrics = eval_topics(topic_terms, docs)
        results.append(('NGRAM', metrics['objective'], metrics, cfg))
    except Exception as e:
        print("N-gram config failed", cfg, e)

# --- BERTopic sweep (optional) ---
if BERTOPIC_TRIALS > 0:
    if 'bertopic' not in globals():
        print("BERTopic sweep skipped (bertopic model not loaded).")
    else:
        try:
            from copy import deepcopy
            base_docs = docs
            for _ in range(BERTOPIC_TRIALS):
                cfg = {
                    'min_topic_size': random.choice([10, 20, 30]),
                    'nr_topics': random.choice([None, 10, 15, 20]),
                    'diversity': random.choice([0.1, 0.3, 0.5]),
                    'top_n_words': random.choice([10, 15, 20]),
                }
                model = deepcopy(bertopic)
                model.top_n_words = cfg['top_n_words']
                topics, _ = model.fit_transform(base_docs)
                topic_info = model.get_topic_info()
                topic_ids = [t for t in topic_info['Topic'].tolist() if t != -1]
                topic_terms = []
                for t in topic_ids:
                    words = [w for w, _ in model.get_topic(t)][:TOPN_TERMS]
                    topic_terms.append(words)
                metrics = eval_topics(topic_terms, base_docs)
                results.append(('BERTopic', metrics['objective'], metrics, cfg))
        except Exception as e:
            print("BERTopic sweep skipped due to error:", e)

# --- Report best configs ---
if results:
    results_sorted = sorted(results, key=lambda x: x[1], reverse=True)
    print("Top results (overall, objective, metrics, config):")
    for entry in results_sorted[:5]:
        method, obj, metrics, cfg = entry
        print("- {}: objective={:.3f}, diversity={:.3f}, overlap={:.3f}, coverage={:.3f}, cfg={}".format(
            method, obj, metrics['diversity'], metrics['overlap'], metrics['coverage'], cfg
        ))

    best_by_method = {}
    for method, obj, metrics, cfg in results:
        if method not in best_by_method or obj > best_by_method[method][0]:
            best_by_method[method] = (obj, metrics, cfg)
    print("\nBest per method:")
    for method, (obj, metrics, cfg) in best_by_method.items():
        print("- {}: objective={:.3f}, diversity={:.3f}, overlap={:.3f}, coverage={:.3f}, cfg={}".format(
            method, obj, metrics['diversity'], metrics['overlap'], metrics['coverage'], cfg
        ))
else:
    print("No results computed; check inputs/configs.")


In [ ]:

# Apply optimized params for LDA / N-gram / BERTopic

# Requires chunks_df['text'] to be present
if 'chunks_df' not in globals():
    raise RuntimeError("Load chunks_df before running this cell.")
texts = chunks_df['text'].fillna('').tolist()
if not texts:
    raise RuntimeError("No text available in chunks_df.")

# LDA with optimized params
lda_vec = CountVectorizer(max_df=0.9, min_df=1, ngram_range=(1, 2))
lda_dtm = lda_vec.fit_transform(texts)
lda_model = LatentDirichletAllocation(
    n_components=10,
    learning_method='batch',
    max_iter=10,
    random_state=42,
)
lda_model.fit(lda_dtm)
lda_terms = lda_vec.get_feature_names_out()
lda_topic_terms = [[lda_terms[i] for i in comp.argsort()[:-11:-1]] for comp in lda_model.components_]
print("Fitted LDA with optimized params. Topic 0 terms:", ', '.join(lda_topic_terms[0]))

# N-gram TF-IDF with optimized params
# N-gram TF-IDF with optimized params
ng_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_df=0.95,
    min_df=1,
    token_pattern=r"(?u)\b\w+\b",
    lowercase=True,
)
ng_tfidf = ng_vec.fit_transform(texts)
if ng_tfidf.shape[1] == 0:
    raise RuntimeError("N-gram TF-IDF vocabulary is empty; lower min_df/max_df or check texts.")
ng_scores = ng_tfidf.sum(axis=0).A1
ng_terms = ng_vec.get_feature_names_out()
ng_top_idx = ng_scores.argsort()[::-1][:20]
ng_top_terms = [ng_terms[i] for i in ng_top_idx]
print("Top 20 n-grams (optimized params):", ng_top_terms)
# BERTopic with optimized params (requires bertopic loaded)
if 'bertopic' in globals():
    from copy import deepcopy
    bert_cfg = {
        'min_topic_size': 20,
        'nr_topics': None,
        'diversity': 0.3,
        'top_n_words': 15,
    }
    bt = deepcopy(bertopic)
    bt.top_n_words = bert_cfg['top_n_words']
    topics, _ = bt.fit_transform(texts)
    info = bt.get_topic_info()
    valid_topics = [t for t in info['Topic'].tolist() if t != -1]
    sample_topic = valid_topics[0] if valid_topics else None
    if sample_topic is not None:
        print(f"BERTopic fitted with optimized params. Sample topic {sample_topic}:", bt.get_topic(sample_topic))
    else:
        print("BERTopic fitted but no valid topics returned.")
else:
    print("BERTopic model not loaded; skipped BERTopic fit.")


In [ ]:
# Fit optimized params for LDA / N-gram / BERTopic and show topics

if 'chunks_df' not in globals():
    raise RuntimeError("Load chunks_df before running this cell.")
texts = chunks_df['text'].fillna('').tolist()
if not texts:
    raise RuntimeError("No text available in chunks_df.")

stop_list = stop_words if 'stop_words' in globals() else None

# LDA fit
lda_vec = CountVectorizer(max_df=0.9, min_df=1, ngram_range=(1, 2), stop_words=stop_list)
lda_dtm = lda_vec.fit_transform(texts)
lda_model = LatentDirichletAllocation(n_components=10, learning_method='batch', max_iter=10, random_state=42)
lda_model.fit(lda_dtm)
lda_terms = lda_vec.get_feature_names_out()
print("LDA topics (top 10 terms):")
for tid, comp in enumerate(lda_model.components_):
    top_idx = comp.argsort()[:-11:-1]
    terms = [lda_terms[i] for i in top_idx]
    print(f"Topic {tid}: {', '.join(terms)}")

# N-gram TF-IDF
ng_vec = TfidfVectorizer(
    ngram_range=(6, 6),
    max_df=0.95,
    min_df=1,
    token_pattern=r"(?u)\b\w+\b",
    lowercase=True,
    stop_words=stop_list,
)
ng_tfidf = ng_vec.fit_transform(texts)
if ng_tfidf.shape[1] == 0:
    raise RuntimeError("N-gram TF-IDF vocabulary is empty; adjust stopwords or df thresholds.")
ng_scores = ng_tfidf.sum(axis=0).A1
ng_terms = ng_vec.get_feature_names_out()
ng_top_idx = ng_scores.argsort()[::-1][:20]
ng_top_terms = [ng_terms[i] for i in ng_top_idx]
print("\nTop 20 n-grams (optimized params):")
for t in ng_top_terms:
    print("-", t)

# BERTopic fit (if model loaded)
if 'bertopic' in globals():
    from copy import deepcopy
    bert_cfg = {'min_topic_size': 20, 'nr_topics': None, 'diversity': 0.3, 'top_n_words': 15}
    bt = deepcopy(bertopic)
    bt.top_n_words = bert_cfg['top_n_words']
    topics, _ = bt.fit_transform(texts)
    info = bt.get_topic_info()
    valid_topics = [t for t in info['Topic'].tolist() if t != -1]
    print("\nBERTopic (optimized params) top topics:")
    for t in valid_topics[:5]:
        words = [w for w, _ in bt.get_topic(t)][:10]
        print(f"Topic {t}: {', '.join(words)}")
else:
    print("\nBERTopic model not loaded; skip BERTopic fit.")


In [ ]:
# Test summary of optimized topics (now also saves to CSV per method)
ensure_retrieval()
ensure_llm()

texts_opt = chunks_df['text'].fillna('').tolist()
stop_list = stop_words if 'stop_words' in globals() else None

ng_records = []
lda_records = []
bt_records = []

# N-gram summaries (optimized)
try:
    ng_vec = TfidfVectorizer(
        ngram_range=(1, 3),
        max_df=0.95,
        min_df=1,
        token_pattern=r"(?u)\b\w+\b",
        lowercase=True,
        stop_words=stop_list,
    )
    ng_tfidf = ng_vec.fit_transform(texts_opt)
    scores = ng_tfidf.sum(axis=0).A1
    terms = ng_vec.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:5]
    opt_ngrams = [terms[i] for i in top_idx]
    for ng in opt_ngrams:
        query = f"Summarize the discussion about '{ng}' in 2-3 sentences."
        hits = search_chunks(query, k=6)
        context = "\n\n".join(
            f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}" for h in hits
        )
        prompt = build_prompt(context, query)
        summary = _gen(prompt)[0]['generated_text']
        general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about {ng}."
        general_summary = _gen(general_prompt)[0]['generated_text']
        ng_records.append({
            'ngram': ng,
            'summary_contextual': summary,
            'summary_general': general_summary,
            'sources': ', '.join(h['chunk_id'] for h in hits),
        })
        print("=" * 80)
        print("N-gram (opt):", ng)
        print("Summary:", summary)
        print("Sources:", ", ".join(h['chunk_id'] for h in hits))
        print("General summary (LLM, not limited to transcript):", general_summary)
    if ng_records:
        pd.DataFrame(ng_records).to_csv('optimized_ngram_topics.csv', index=False)
        print('Saved optimized n-gram topics to optimized_ngram_topics.csv')
except Exception as exc:
    print("Optimized n-gram summaries skipped:", exc)

# LDA summaries (optimized)
try:
    lda_vec = CountVectorizer(max_df=0.9, min_df=1, ngram_range=(1, 2), stop_words=stop_list)
    lda_dtm = lda_vec.fit_transform(texts_opt)
    lda_model = LatentDirichletAllocation(n_components=10, learning_method='batch', max_iter=8, random_state=42)
    lda_model.fit(lda_dtm)
    lda_terms = lda_vec.get_feature_names_out()
    for t in range(lda_model.n_components):
        top_idx = lda_model.components_[t].argsort()[:-7:-1]
        term_str = ", ".join(lda_terms[i] for i in top_idx)
        query = f"Summarize the discussion around these keywords: {term_str}. Keep it to 2-3 sentences."
        hits = search_chunks(query, k=6)
        context = "\n\n".join(
            f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}" for h in hits
        )
        prompt = build_prompt(context, query)
        summary = _gen(prompt)[0]['generated_text']
        general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about these keywords: {term_str}."
        general_summary = _gen(general_prompt)[0]['generated_text']
        lda_records.append({
            'topic_id': t,
            'keywords': term_str,
            'summary_contextual': summary,
            'summary_general': general_summary,
            'sources': ', '.join(h['chunk_id'] for h in hits),
        })
        print("=" * 80)
        print(f"LDA (opt) Topic {t} terms:", term_str)
        print("Summary:", summary)
        print("Sources:", ", ".join(h['chunk_id'] for h in hits))
        print("General summary (LLM, not limited to transcript):", general_summary)
    if lda_records:
        pd.DataFrame(lda_records).to_csv('optimized_lda_topics.csv', index=False)
        print('Saved optimized LDA topics to optimized_lda_topics.csv')
except Exception as exc:
    print("Optimized LDA summaries skipped:", exc)

# BERTopic summaries (optimized params) if model loaded
try:
    if 'bertopic' in globals():
        from copy import deepcopy
        bt = deepcopy(bertopic)
        bt.top_n_words = 15
        topics, _ = bt.fit_transform(texts_opt)
        info = bt.get_topic_info()
        valid_topics = [t for t in info['Topic'].tolist() if t != -1]
        for t in valid_topics:
            terms_bt = [w for w, _ in bt.get_topic(t)][:6]
            term_str = ", ".join(terms_bt)
            query = f"Summarize the discussion around these keywords: {term_str}. Keep it to 2-3 sentences."
            hits = search_chunks(query, k=6)
            context = "\n\n".join(
                f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}" for h in hits
            )
            prompt = build_prompt(context, query)
            summary = _gen(prompt)[0]['generated_text']
            general_prompt = f"Provide a general (not transcript-bound) 2-3 sentence summary about these keywords: {term_str}."
            general_summary = _gen(general_prompt)[0]['generated_text']
            bt_records.append({
                'topic_id': t,
                'keywords': term_str,
                'summary_contextual': summary,
                'summary_general': general_summary,
                'sources': ', '.join(h['chunk_id'] for h in hits),
            })
            print("=" * 80)
            print(f"BERTopic (opt) {t} terms:", term_str)
            print("Summary:", summary)
            print("Sources:", ", ".join(h['chunk_id'] for h in hits))
            print("General summary (LLM, not limited to transcript):", general_summary)
        if bt_records:
            pd.DataFrame(bt_records).to_csv('optimized_bertopic_topics.csv', index=False)
            print('Saved optimized BERTopic topics to optimized_bertopic_topics.csv')
    else:
        print("Optimized BERTopic summaries skipped: bertopic model not loaded.")
except Exception as exc:
    print("Optimized BERTopic summaries skipped:", exc)


In [ ]:

# Chosen Method
# Ultimate BERTopic summary from optimized_bertopic_topics.csv

ensure_llm()

csv_path = Path('optimized_bertopic_topics.csv') # Change this if necessary
if not csv_path.exists():
    print('optimized_bertopic_topics.csv not found; run the optimized BERTopic export cell first.')
else:
    df = pd.read_csv(csv_path)
    if df.empty:
        print('optimized_bertopic_topics.csv is empty.')
    else:
        all_keywords = df['keywords'].dropna().tolist()
        combined_keywords = '; '.join(all_keywords)
        print('Combined keywords:')
        print(combined_keywords)
        combined_context = '\n'.join(df['summary_contextual'].dropna().tolist())
        prompt = (
            "You are a research analyst."
            "Summarize all BERTopic topics in 5-7 concise sentences. Avoid repetition."
            "Use only the provided summaries and keywords; do not add outside info."
            f"Keywords: {combined_keywords}\n\n"
            f"Summaries:\n{combined_context}\n\n"
            "Ultimate summary:"
        )
        prompt_str = ''.join(prompt)
        ultimate = _gen(prompt_str, no_repeat_ngram_size=3)[0]['generated_text']
        print('Summary using genAI:')
        print(ultimate)


In [ ]:
# new base
# Ultimate BERTopic summary from bertopic_topics.xlsx

ensure_llm()

csv_path = Path('bertopic_topics.xlsx')  # Change this if necessary
if not csv_path.exists():
    print('bertopic_topics.xlsx not found; run the BERTopic export cell first.')
else:
    df = pd.read_excel(csv_path)
    if df.empty:
        print('bertopic_topics.xlsx is empty.')
    else:
        def parse_rep(rep):
            if isinstance(rep, str):
                rep_str = rep.strip()
                if rep_str.startswith('[') and rep_str.endswith(']'):
                    try:
                        parsed = ast.literal_eval(rep_str)
                        if isinstance(parsed, list):
                            return [str(x).strip() for x in parsed if str(x).strip()]
                    except (ValueError, SyntaxError):
                        pass
                # fallback: comma split
                return [r.strip(" '\"") for r in rep_str.split(',') if r.strip()]
            if isinstance(rep, (list, tuple)):
                return [str(x).strip() for x in rep if str(x).strip()]
            return []

        topic_lines = []
        for _, row in df.iterrows():
            name = str(row.get('Name', '')).strip()
            topic_id = str(row.get('Topic', '')).strip()
            label = name or f"Topic {topic_id}"
            rep_list = parse_rep(row.get('Representation', ''))
            if rep_list:
                rep_str = ', '.join(rep_list[:10])
                topic_lines.append(f"{label} covers {rep_str}.")
            else:
                topic_lines.append(f"{label}.")

        if not topic_lines:
            print('No topic representations found to summarize.')
        else:
            print('Topics for summary:')
            print('\n'.join(topic_lines))

            prompt = (
                "You are a research analyst writing a short narrative summary."
                "Write 5-7 complete sentences in one paragraph."
                "Synthesize across topics; do not list keywords or repeat labels."
                "Do not use bullet points, brackets, or colons."
                "Use only the topic descriptions below; do not add outside info."
                "\n\nTopic descriptions:\n"
                + "\n".join(topic_lines)
                + "\n\nSummary:"
            )
            ultimate = _gen(prompt, no_repeat_ngram_size=3, max_new_tokens=240)[0]['generated_text']

            def needs_rewrite(text):
                if text.count('.') < 2:
                    return True
                bad_tokens = ['[', ']', ':', ';', '\n-']
                return any(t in text for t in bad_tokens)

            if needs_rewrite(ultimate):
                rewrite_prompt = (
                    "Rewrite the text below into 5-7 complete sentences in one paragraph."
                    "No lists, no brackets, no colons."
                    "Focus on themes and integrate ideas."
                    "\n\nText:\n"
                    + ultimate
                    + "\n\nRewritten summary:"
                )
                ultimate = _gen(rewrite_prompt, no_repeat_ngram_size=3, max_new_tokens=240)[0]['generated_text']

            print('Summary using genAI:')
            print(ultimate)


In [ ]:
# Hyperparameter sweep for RAG eval (SIM_THRESHOLD, K, TEMP, TOP_P)

SIM_THRESHOLDS = [0.65, 0.7, 0.75]
K_VALUES = [4, 6, 8]
TEMP_VALUES = [0.3, 0.5]
TOP_P_VALUES = [0.85, 0.9]
MAX_QUESTIONS = None  # set to an int (e.g., 10) to subsample for faster sweeps
EVAL_CSV = 'eval_rag_results.csv'

ensure_retrieval()
ensure_llm()

qe_df = pd.read_csv(EVAL_CSV)
if 'question' not in qe_df.columns:
    raise RuntimeError("Eval CSV must have a 'question' column.")
if MAX_QUESTIONS:
    qe_df = qe_df.head(MAX_QUESTIONS).copy()

retrieval_cache = {}

def evaluate_combo(sim_threshold, k, temp, top_p):
    rows = []
    for _, row in qe_df.iterrows():
        q = str(row['question']).strip()
        if not q:
            continue
        cache_key = (q, k)
        if cache_key not in retrieval_cache:
            retrieval_cache[cache_key] = search_chunks(q, k=k)
        hits = retrieval_cache[cache_key]
        context = "\n\n".join(
            f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}" for h in hits
        )
        prompt = build_prompt(context, q)
        ans = _gen(prompt, temperature=temp, top_p=top_p)[0]['generated_text']
        ref = (
            str(row['reference']).strip()
            if 'reference' in qe_df.columns and pd.notna(row['reference'])
            else ''
        )
        if ref:
            sim = SequenceMatcher(None, ans.lower(), ref.lower()).ratio()
            correct = sim >= sim_threshold
        else:
            sim = None
            correct = 'not enough context' not in ans.lower()
        rows.append({
            'question': q,
            'answer': ans,
            'reference': ref,
            'similarity': sim,
            'correct': correct,
            'k': k,
            'temp': temp,
            'top_p': top_p,
            'sim_threshold': sim_threshold,
        })
    score_df = pd.DataFrame(rows)
    correct_count = int(score_df['correct'].sum()) if not score_df.empty else 0
    avg_sim = (
        float(score_df['similarity'].dropna().mean())
        if not score_df.empty and score_df['similarity'].notna().any()
        else None
    )
    return score_df, correct_count, avg_sim

results = []
for iteration, (sim_threshold, k, temp, top_p) in enumerate(
    itertools.product(SIM_THRESHOLDS, K_VALUES, TEMP_VALUES, TOP_P_VALUES), start=1
):
    score_df, correct_count, avg_sim = evaluate_combo(sim_threshold, k, temp, top_p)
    total = len(score_df)
    label = f"sim={sim_threshold},k={k},T={temp},p={top_p}"
    results.append({
        'iteration': iteration,
        'sim_threshold': sim_threshold,
        'k': k,
        'temp': temp,
        'top_p': top_p,
        'correct': correct_count,
        'total': total,
        'accuracy': correct_count / total if total else 0.0,
        'avg_similarity': avg_sim,
        'label': label,
    })
    print(
        f"iter={iteration} {label} -> correct {correct_count}/{total}, avg_sim={avg_sim}"
    )

results_df = pd.DataFrame(results)
if results_df.empty:
    best_params = {}
else:
    results_df = results_df.sort_values(by=['correct', 'avg_similarity'], ascending=False, na_position='last').reset_index(drop=True)
    best_params = results_df.iloc[0].to_dict()
    print("\nBest params (by correct count, then avg_similarity):")
    print(best_params)

if not results_df.empty:
    x = range(len(results_df))
    plt.figure(figsize=(10, 4))
    plt.plot(x, results_df['correct'], marker='o')
    plt.title('Hyperparameter sweep (correct count)')
    plt.xlabel('Hyperparameter combo')
    plt.ylabel('Correct')
    plt.xticks(x, results_df['label'], rotation=45, ha='right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

results_df, best_params


In [ ]:
# Eval scoring: baseline RAG 
# Expects a CSV with a 'question' column; optional 'reference' column for target answers


EVAL_CSV = 'eval_rag_results.csv'  # change if your eval questions live elsewhere
SIM_THRESHOLD = 0.7  # similarity threshold if reference answers exist
K = DEFAULT_TOP_K
TEMP = 0.4
TOP_P = 0.9

# Ensure retrieval + LLM are ready
ensure_retrieval()
ensure_llm()

# Load eval questions
qe_df = pd.read_csv(EVAL_CSV)
if 'question' not in qe_df.columns:
    raise RuntimeError("Eval CSV must have a 'question' column.")

rows = []
for _, row in qe_df.iterrows():
    q = str(row['question']).strip()
    if not q:
        continue
    hits = search_chunks(q, k=K)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, q)
    ans = _gen(prompt, temperature=TEMP, top_p=TOP_P)[0]['generated_text']
    ref = str(row['reference']).strip() if 'reference' in qe_df.columns and pd.notna(row['reference']) else ''
    sim = SequenceMatcher(None, ans.lower(), ref.lower()).ratio() if ref else None
    if ref:
        correct = sim >= SIM_THRESHOLD
    else:
        correct = not ('not enough context' in ans.lower())
    rows.append({
        'question': q,
        'answer': ans,
        'reference': ref,
        'similarity': sim,
        'correct': correct,
        'sources': ', '.join(h['chunk_id'] for h in hits),
    })

score_df = pd.DataFrame(rows)

total = len(score_df)
answered = sum(score_df['correct']) if not score_df.empty else 0
print(f"Total: {total} | Correct (heuristic): {answered}")
score_df.head()

export_path = 'eval_rag_scored.xlsx'
score_df.to_excel(export_path, index=False)
print(f'Saved evaluation scores to: {export_path}')


In [ ]:

# Single-method RAG QA widget (direct RAG only)

ensure_retrieval()
ensure_llm()  # auto-loads flan-t5-large


def run_rag(question, k):
    hits = search_chunks(question, k=k)
    context = "\n\n".join(
        f"[{h['chunk_id']}] {textwrap.shorten(h['text'], width=500, placeholder='...')}"
        for h in hits
    )
    prompt = build_prompt(context, question)
    answer = _gen(prompt)[0]['generated_text']
    return answer, hits


question_box = widgets.Textarea(
    value="",
    description="Question",
    placeholder="Type a question",
    layout=widgets.Layout(width="100%", height="80px"),
)
k_slider = widgets.IntSlider(value=6, min=2, max=10, step=1, description="Top k")
run_button = widgets.Button(description="Run", button_style="primary")
output = widgets.Output()


def on_run_clicked(_):
    q = question_box.value.strip()
    output.clear_output()
    if not q:
        with output:
            print("Enter a question to run RAG.")
        return
    try:
        ans, hits = run_rag(q, k_slider.value)
        with output:
            display(Markdown(f"**Question:** {q}"))
            display(Markdown(f"**Answer:** {ans}"))
            if hits:
                print("Sources:")
                for h in hits:
                    snippet = textwrap.shorten(h.get('text', ''), width=180, placeholder='...')
                    print(f"- {h.get('chunk_id')}: {snippet}")
    except Exception as exc:
        with output:
            print(f"Error: {exc}")


run_button.on_click(on_run_clicked)
controls = widgets.VBox([question_box, k_slider, run_button])
display(widgets.VBox([controls, output]))
